# FINANCE 384 Assignment 1 – Part A

## A.2 Pooled OLS Benchmark

This notebook estimates the transparent pooled ordinary least squares (OLS) benchmark for forecasting next-month stock excess returns.

The notebook uses the same A.1 data preparation and base predictor information that will also be supplied to the richer model:

- next-month excess-return target constructed through a calendar-month stock join;
- 18 admissible numeric characteristics plus FF49 industry membership;
- contemporaneous FF49 industry-month median imputation;
- same-month market median fallback;
- missingness indicators;
- training-sample standardisation of continuous predictors;
- one-hot encoding of FF49 industry membership.

The pooled OLS benchmark is estimated on the **combined training and validation samples (January 1990–December 2018)**. The **January 2019–November 2022 test sample remains untouched for final out-of-sample evaluation**.


In [1]:
# A.2.1 Imports and file locations

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"

TARGET = "target_ret_excess_tp1"


In [2]:
# A.2.2 Load supplied data

panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print("Date range:",
      panel["date"].min().date(),
      "to",
      panel["date"].max().date())

print("Unique stocks:", panel["permno"].nunique())
print("Duplicate stock-month rows:",
      panel.duplicated(["permno", "date"]).sum())


Panel shape: (198298, 27)
Date range: 1990-01-31 to 2022-12-30
Unique stocks: 1252
Duplicate stock-month rows: 0


In [3]:
# A.2.3 Define the common predictor set

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [
    x for x in numeric_predictors
    if x != "down_market"
]

binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]

identifier_columns = [
    "date",
    "permno",
    "ticker",
    "comnam",
    "gvkey",
]

print("Numeric predictors:", len(numeric_predictors))
print("Continuous predictors:", len(continuous_predictors))
print("Binary predictors:", binary_predictors)
print("Industry predictor:", categorical_predictors)


Numeric predictors: 18
Continuous predictors: 17
Binary predictors: ['down_market']
Industry predictor: ['ff49_code']


### Reproduce the revised A.1 target construction

The target \(r^e_{i,t+1}\) is attached using a calendar-month join so that stocks that leave and later re-enter the S&P 500 are not incorrectly linked across gaps.


In [4]:
# A.2.4 Construct the next-calendar-month target

panel = panel.sort_values(
    ["permno", "date"]
).reset_index(drop=True)

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[
        ["permno", "month", "ret_excess_t"]
    ]
    .rename(
        columns={
            "ret_excess_t": TARGET
        }
    )
    .assign(
        month=lambda df: df["month"] - 1
    )
)

panel = panel.merge(
    next_month_return,
    on=["permno", "month"],
    how="left",
    validate="one_to_one",
)

print("Rows with valid next-month target:",
      panel[TARGET].notna().sum())


Rows with valid next-month target: 197016


### Reproduce the revised A.1 missing-data treatment

Missing continuous characteristics are filled using the same-month FF49 industry median, with the same-month market median as a fallback. Missingness indicators are created before imputation and retained as common predictors for both OLS and the richer model.


In [5]:
# A.2.5 Create missingness indicators

missing_characteristics = [
    col for col in continuous_predictors
    if panel[col].isna().any()
]

missing_indicator_columns = []

for col in missing_characteristics:
    indicator = f"{col}_was_missing"
    panel[indicator] = panel[col].isna().astype(int)
    missing_indicator_columns.append(indicator)

print("Missingness indicators created:")
print(missing_indicator_columns)


Missingness indicators created:
['bm_was_missing', 'mom12_2_was_missing', 'vol12_was_missing', 'beta60_was_missing', 'ivol60_was_missing', 'divyield_was_missing', 'gross_profit_was_missing', 'roe_was_missing', 'asset_growth_was_missing', 'leverage_was_missing', 'accruals_was_missing']


In [6]:
# A.2.6 Apply industry-month median imputation with market-month fallback

for col in continuous_predictors:

    industry_month_median = (
        panel
        .groupby(
            ["month", "ff49_code"]
        )[col]
        .transform("median")
    )

    market_month_median = (
        panel
        .groupby("month")[col]
        .transform("median")
    )

    panel[col] = (
        panel[col]
        .fillna(industry_month_median)
        .fillna(market_month_median)
    )

remaining_missing = panel[continuous_predictors].isna().sum()

print(
    "Remaining missing continuous predictor values:",
    int(remaining_missing.sum())
)
print(
    "Missing down_market values:",
    int(panel["down_market"].isna().sum())
)
print(
    "Missing ff49_code values:",
    int(panel["ff49_code"].isna().sum())
)


Remaining missing continuous predictor values: 0
Missing down_market values: 0
Missing ff49_code values: 0


In [7]:
# A.2.7 Keep valid forecast observations and apply the fixed chronological split

analysis = panel.loc[
    panel[TARGET].notna()
].copy()

train = analysis.loc[
    (analysis["date"] >= "1990-01-01")
    & (analysis["date"] <= "2014-12-31")
].copy()

validation = analysis.loc[
    (analysis["date"] >= "2015-01-01")
    & (analysis["date"] <= "2018-12-31")
].copy()

test = analysis.loc[
    (analysis["date"] >= "2019-01-01")
    & (analysis["date"] <= "2022-11-30")
].copy()

train_valid = (
    pd.concat(
        [train, validation],
        axis=0
    )
    .sort_values(
        ["date", "permno"]
    )
    .reset_index(drop=True)
)

sample_summary = pd.DataFrame({
    "Sample": [
        "Training",
        "Validation",
        "OLS estimation (Train + Validation)",
        "Test",
    ],
    "Months": [
        train["month"].nunique(),
        validation["month"].nunique(),
        train_valid["month"].nunique(),
        test["month"].nunique(),
    ],
    "Stock-month rows": [
        len(train),
        len(validation),
        len(train_valid),
        len(test),
    ],
})

sample_summary


,Sample,Months,Stock-month rows
0,Training,300,149334
1,Validation,48,24051
2,OLS estimation (Train + Validation),348,173385
3,Test,47,23631


In [8]:
# A.2.8 Define final common base predictors

raw_feature_columns = (
    continuous_predictors
    + binary_predictors
    + missing_indicator_columns
    + categorical_predictors
)

print("Raw predictor columns:", len(raw_feature_columns))

for feature in raw_feature_columns:
    print("-", feature)


Raw predictor columns: 30
- size
- bm
- mom12_2
- vol12
- beta60
- ivol60
- turnover
- dollar_volume
- amihud_illiq
- divyield
- gross_profit
- roe
- asset_growth
- leverage
- accruals
- mkt_12m
- mkt_vol_12m
- down_market
- bm_was_missing
- mom12_2_was_missing
- vol12_was_missing
- beta60_was_missing
- ivol60_was_missing
- divyield_was_missing
- gross_profit_was_missing
- roe_was_missing
- asset_growth_was_missing
- leverage_was_missing
- accruals_was_missing
- ff49_code


### Common preprocessing

Continuous predictors are standardised using training-sample mean and standard deviation.

`down_market` and missingness indicators are passed through unchanged.

FF49 industry membership is one-hot encoded with one reference category omitted.

Importantly, the preprocessing transformation is fitted on the **training sample only**, then applied unchanged to training + validation and test observations.


In [9]:
# A.2.9 Build and fit the common preprocessing transformation

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_predictors,
        ),
        (
            "binary",
            "passthrough",
            binary_predictors
            + missing_indicator_columns,
        ),
        (
            "industry",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_predictors,
        ),
    ],
    remainder="drop",
)

X_train_raw = train[raw_feature_columns].copy()
X_validation_raw = validation[raw_feature_columns].copy()
X_train_valid_raw = train_valid[raw_feature_columns].copy()
X_test_raw = test[raw_feature_columns].copy()

y_train = train[TARGET].to_numpy()
y_validation = validation[TARGET].to_numpy()
y_train_valid = train_valid[TARGET].to_numpy()
y_test = test[TARGET].to_numpy()

# Fit preprocessing using the training period only.
preprocessor.fit(X_train_raw)

X_train = preprocessor.transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)
X_train_valid = preprocessor.transform(X_train_valid_raw)
X_test = preprocessor.transform(X_test_raw)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)
print("OLS train+validation matrix:", X_train_valid.shape)
print("Test matrix:", X_test.shape)


Training matrix: (149334, 75)
Validation matrix: (24051, 75)
OLS train+validation matrix: (173385, 75)
Test matrix: (23631, 75)


In [10]:
# A.2.10 Retrieve transformed feature names

feature_names = preprocessor.get_feature_names_out()

print(
    "Number of transformed predictors:",
    len(feature_names)
)

pd.Series(
    feature_names,
    name="Transformed predictor"
).head(30)


Number of transformed predictors: 75


0                     continuous__size
1                       continuous__bm
2                  continuous__mom12_2
3                    continuous__vol12
4                   continuous__beta60
5                   continuous__ivol60
6                 continuous__turnover
7            continuous__dollar_volume
8             continuous__amihud_illiq
9                 continuous__divyield
10            continuous__gross_profit
11                     continuous__roe
12            continuous__asset_growth
13                continuous__leverage
14                continuous__accruals
15                 continuous__mkt_12m
16             continuous__mkt_vol_12m
17                 binary__down_market
18              binary__bm_was_missing
19         binary__mom12_2_was_missing
20           binary__vol12_was_missing
21          binary__beta60_was_missing
22          binary__ivol60_was_missing
23        binary__divyield_was_missing
24    binary__gross_profit_was_missing
25             binary__ro

## Estimate the pooled OLS benchmark

The benchmark model is

\[
r^e_{i,t+1}
=
\alpha
+
\beta' z_{i,t}
+
\varepsilon_{i,t+1}.
\]

The OLS coefficients are estimated using the combined training and validation observations. No test observations are used in model estimation.


In [11]:
# A.2.11 Fit pooled OLS on training + validation

ols_model = LinearRegression(
    fit_intercept=True
)

ols_model.fit(
    X_train_valid,
    y_train_valid
)

print("OLS fitted successfully.")
print("OLS estimation observations:", len(y_train_valid))
print("Intercept:", ols_model.intercept_)
print("Number of coefficients:", len(ols_model.coef_))


OLS fitted successfully.
OLS estimation observations: 173385
Intercept: 0.010471502618806649
Number of coefficients: 75


### Retain untouched test predictions for later evaluation

Formal RMSE, monthly Spearman rank correlation, and economic portfolio evaluation are deferred to A.5 and A.6. This notebook only retains the OLS test forecasts needed for those later tasks.


In [12]:
# A.2.12 Generate OLS test predictions

ols_test_pred = ols_model.predict(
    X_test
)

ols_test_predictions = test[
    [
        "date",
        "permno",
        "ticker",
        TARGET,
    ]
].copy()

ols_test_predictions[
    "ols_pred_excess_return_tp1"
] = ols_test_pred

print("Test predictions generated:",
      len(ols_test_predictions))

print(
    "Missing OLS test predictions:",
    int(
        ols_test_predictions[
            "ols_pred_excess_return_tp1"
        ].isna().sum()
    )
)

ols_test_predictions.head()


Test predictions generated: 23631
Missing OLS test predictions: 0


,date,permno,ticker,target_ret_excess_tp1,ols_pred_excess_return_tp1
587,2019-01-31,10104,ORCL,0.036026,0.002192
588,2019-02-28,10104,ORCL,0.028409,0.009638
589,2019-03-29,10104,ORCL,0.032531,0.007082
590,2019-04-30,10104,ORCL,-0.087587,0.012496
591,2019-05-31,10104,ORCL,0.124089,0.013426


### Optional coefficient audit

The following table is retained only as a transparency check. Coefficient-level inference is not the assessed focus of A.2; the benchmark's predictive performance is evaluated later.


In [13]:
# A.2.13 Coefficient audit

coefficient_table = pd.DataFrame({
    "Feature": feature_names,
    "OLS coefficient": ols_model.coef_,
})

coefficient_table["Absolute coefficient"] = (
    coefficient_table[
        "OLS coefficient"
    ].abs()
)

coefficient_table = (
    coefficient_table
    .sort_values(
        "Absolute coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

coefficient_table.head(20)


,Feature,OLS coefficient,Absolute coefficient
0,industry__ff49_code_29,-0.013723,0.013723
1,industry__ff49_code_16,-0.012324,0.012324
2,industry__ff49_code_20,-0.012065,0.012065
3,industry__ff49_code_7,0.011118,0.011118
4,binary__divyield_was_missing,-0.010937,0.010937
5,binary__asset_growth_was_missing,0.008589,0.008589
6,binary__gross_profit_was_missing,-0.008328,0.008328
7,continuous__bm,0.007190,0.007190
8,industry__ff49_code_31,-0.006950,0.006950
9,industry__ff49_code_36,0.006563,0.006563


In [14]:
# A.2.14 Final protocol audit

print("A.2 PROTOCOL AUDIT")
print("-" * 45)

print(
    "OLS estimation sample:",
    "Training + Validation"
)

print(
    "OLS estimation observations:",
    len(train_valid)
)

print(
    "Test observations:",
    len(test)
)

print(
    "Training end:",
    train["date"].max().date()
)

print(
    "Validation end:",
    validation["date"].max().date()
)

print(
    "Test start:",
    test["date"].min().date()
)

print(
    "\nMissing transformed train+validation values:",
    int(np.isnan(X_train_valid).sum())
)

print(
    "Missing transformed test values:",
    int(np.isnan(X_test).sum())
)

print(
    "Missing OLS test predictions:",
    int(np.isnan(ols_test_pred).sum())
)


A.2 PROTOCOL AUDIT
---------------------------------------------
OLS estimation sample: Training + Validation
OLS estimation observations: 173385
Test observations: 23631
Training end: 2014-12-31
Validation end: 2018-12-31
Test start: 2019-01-31

Missing transformed train+validation values: 0
Missing transformed test values: 0
Missing OLS test predictions: 0


## A.2 Summary

A pooled OLS regression is used as the transparent benchmark for next-month stock excess-return prediction.

The benchmark uses the same revised A.1 base predictor information and preprocessing framework intended for the richer model. Continuous characteristics are standardised using training-sample parameters, missingness indicators and `down_market` are retained as binary inputs, and FF49 industry membership is dummy encoded.

In accordance with the assignment protocol, pooled OLS is estimated on the combined January 1990–December 2018 training and validation sample. The January 2019–November 2022 test period is excluded from model estimation and is reserved for final out-of-sample evaluation.
